# 第 4 章 過学習・未学習と正則化

多項式回帰で次数を変えながら、訓練誤差とテスト誤差の差（汎化ギャップ）を観察し、L1 / L2 正則化の効果を確かめます。

対応する記事: [第 4 章 過学習・未学習と正則化（Python 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/python/ch04.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch04_regularization import *

## データセット

`y = 2x + 3` に小さなノイズを乗せた 10 点です。**真の関係は 1 次関数** なので、高次のモデルは過学習するはずです。

In [2]:
features = [-1.5, -1.2, -0.9, -0.6, -0.3, 0.0, 0.3, 0.6, 0.9, 1.2]
labels = [0.08, 0.32, 1.07, 1.63, 2.54, 3.11, 3.84, 3.95, 4.75, 5.12]

train_x, train_y, test_x, test_y = train_test_split(features, labels, test_ratio=0.3, seed=0)
print("訓練", [round(x, 1) for x in train_x])
print("テスト", [round(x, 1) for x in test_x])

訓練 [0.0, -0.6, -0.3, -0.9, -1.5, 1.2, 0.3]
テスト [0.6, 0.9, -1.2]


## 次数を上げるとどうなるか

**訓練誤差は下がるのに、汎化ギャップ（テスト誤差 − 訓練誤差）は広がります。** これが過学習です。

In [3]:
print(f"{'次数':>4} {'訓練 RMSE':>10} {'テスト RMSE':>12} {'ギャップ':>10}")
for degree in [1, 3, 5]:
    m = polynomial_regression(train_x, train_y, degree=degree, learning_rate=0.01, epochs=20000)
    train_error = model_rmse(m, train_x, train_y)
    test_error = model_rmse(m, test_x, test_y)
    print(f"{degree:>4} {train_error:>10.4f} {test_error:>12.4f} {test_error - train_error:>10.4f}")

  次数    訓練 RMSE     テスト RMSE       ギャップ
   1     0.1736       0.2200     0.0464


   3     0.0550       0.2988     0.2438


   5     0.0601       0.2947     0.2346


## 正則化を掛ける

5 次モデルに λ = 0.01 の正則化を掛けます。**訓練誤差は悪化しますが、テスト誤差は改善します。** 訓練データへの当てはまりをわざと諦めて、未知データへの当てはまりを買っています。

In [4]:
print(f"{'正則化':<8} {'訓練':>8} {'テスト':>8} {'重みの合計':>10}")
for kind in [Regularization.NONE, Regularization.L1, Regularization.L2]:
    strength = 0.0 if kind is Regularization.NONE else 0.01
    m = polynomial_regression(train_x, train_y, degree=5, learning_rate=0.01, epochs=20000,
                              kind=kind, strength=strength)
    print(f"{kind.value:<8} {model_rmse(m, train_x, train_y):>8.4f} "
          f"{model_rmse(m, test_x, test_y):>8.4f} {weight_magnitude(m):>10.3f}")

正則化            訓練      テスト      重みの合計
none       0.0601   0.2947      2.747


l1         0.0743   0.2231      2.422


l2         0.1315   0.1239      2.820


## L1 は重みを 0 にする

学習された 5 つの重みを並べます。**L1 は不要な次数の重みをほぼ 0 まで押し下げます**（スパース性）。L2 は全体をなだらかに縮めるだけで、0 にはしません。

In [5]:
for kind in [Regularization.NONE, Regularization.L1, Regularization.L2]:
    strength = 0.0 if kind is Regularization.NONE else 0.01
    m = polynomial_regression(train_x, train_y, degree=5, learning_rate=0.01, epochs=20000,
                              kind=kind, strength=strength)
    print(f"{kind.value:<6}", "  ".join(f"{w:8.4f}" for w in m.weights))

none     2.2274   -0.1567    0.0930   -0.0881   -0.1821
l1       2.1233   -0.2141   -0.0003   -0.0005   -0.0839


l2       1.7088   -0.4038    0.4203    0.1182   -0.1689


## 試してみる

正則化の強さを変えるとどうなるでしょうか。**強すぎるとかえって悪化します。** 直感に反しますが、重みを潰しすぎると境界の位置そのものが崩れるためです。

In [6]:
for strength in [0.0, 0.005, 0.01, 0.05, 0.2]:
    m = polynomial_regression(train_x, train_y, degree=5, learning_rate=0.01, epochs=20000,
                              kind=Regularization.L2, strength=strength)
    print(f"λ = {strength:<6} テスト RMSE {model_rmse(m, test_x, test_y):.4f}")

λ = 0.0    テスト RMSE 0.2947


λ = 0.005  テスト RMSE 0.1442
λ = 0.01   テスト RMSE 0.1239


λ = 0.05   テスト RMSE 0.4740


λ = 0.2    テスト RMSE 0.9347
